# View LSSTCam DeepCoadd Mosaic in Matplotlib

- author Sylvie Dagoret-Campagne
- creation date 2026-03-26
- LSST pipelines : w_2025_10

## Import

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# import lsst.daf.butler as dafButler
from lsst.daf.butler import Butler

import lsst.geom as geom
from lsst.geom import SpherePoint, degrees


from lsst.skymap import PatchInfo, Index2D

from lsst.afw.math import binImage

In [ ]:
import gc

In [ ]:
from astropy.visualization import ZScaleInterval

In [ ]:
def dataset_type_exists(butler, name):
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False

In [ ]:
def get_first_existing_dataset(butler, dataset_names, dataId, collections=None):
    for name in dataset_names:
        try:
            obj = butler.get(name, dataId, collections=collections)
            print(f"✔ Found dataset: {name}")
            return obj, name
        except Exception:
            continue

    raise ValueError("No valid dataset found among candidates.")

## Config

In [ ]:
# The output repo is tagged with the Jira ticket number "DM-40356":
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

# bad : crash collection = 'LSSTComCam/runs/DRP/DP1/w_2025_08/DM-49029'

# bad : collection = "LSSTComCam/runs/DRP/20241101_20241211/w_2024_51/DM-48233"

# not working perhaps because I am using w_2025_10 version
# bad : no ccd visit collection = "LSSTComCam/runs/DRP/DP1/w_2025_14/DM-49864"
# bad : no ccd visit collection = 'LSSTComCam/runs/DRP/DP1/w_2025_15/DM-50050'
# bad : no cce visit collection = 'LSSTComCam/runs/DRP/DP1/w_2025_14/DM-49864'
# bad : no cce visit collection collection = 'LSSTComCam/runs/DRP/DP1/w_2025_13/DM-49751'
instrument = "LSSTCam"
skymapName = "lsst_cells_v2"
where_clause = "instrument = '" + instrument + "'"
collectionStr = collection[-1].replace("/", "_")
BANDSEL = "r"  # Most fields were observed in red filter

In [ ]:
# Initialize the butler repo:
butler = Butler(repo, collections=collection)
registry = butler.registry

In [ ]:
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)

In [ ]:
camera = butler.get("camera", collections=collection, instrument=instrument)

In [ ]:
# print(camera.getName(),camera.getNameMap())

## List of Sky field of interest

In [ ]:
lsstcam_targets = {}
lsstcam_targets["47 Tuc"] = {"field_name": "47 Tuc Globular Cluster", "ra": 6.02, "dec": -72.08}
lsstcam_targets["Rubin SV 38 7"] = {"field_name": "Low Ecliptic Latitude Field", "ra": 37.86, "dec": 6.98}
lsstcam_targets["Fornax dSph"] = {"field_name": "Fornax Dwarf Spheroidal Galaxy", "ra": 40.0, "dec": -34.45}
lsstcam_targets["ECDFS"] = {"field_name": "Extended Chandra Deep Field South", "ra": 53.13, "dec": -28.10}
lsstcam_targets["EDFS"] = {"field_name": "Euclid Deep Field South", "ra": 59.10, "dec": -48.73}
lsstcam_targets["Rubin SV 95 -25"] = {"field_name": "Low Galactic Latitude Field", "ra": 95.00, "dec": -25.0}
lsstcam_targets["Seagull"] = {"field_name": "Seagull Nebula", "ra": 106.23, "dec": -10.51}

In [ ]:
lsstcam_fields = {}

# =========================
# Deep Drilling Fields (LSSTCam)
# =========================
lsstcam_fields["COSMOS"] = {"field_name": "COSMOS Deep Drilling Field", "ra": 150.11, "dec": 2.23}

lsstcam_fields["XMM-LSS"] = {"field_name": "XMM-LSS Deep Drilling Field", "ra": 35.57, "dec": -4.82}

lsstcam_fields["ELAIS-S1"] = {"field_name": "ELAIS-S1 Deep Drilling Field", "ra": 9.45, "dec": -44.02}

lsstcam_fields["ECDFS"] = {"field_name": "Extended Chandra Deep Field South", "ra": 52.98, "dec": -28.12}

# Euclid Deep Field South = 2 pointings LSSTCam
lsstcam_fields["EDFS_a"] = {"field_name": "Euclid Deep Field South (pointing a)", "ra": 58.90, "dec": -49.32}

lsstcam_fields["EDFS_b"] = {"field_name": "Euclid Deep Field South (pointing b)", "ra": 63.60, "dec": -47.60}

In [ ]:
def generate_wfd_pointings(skymap):
    fields = {}

    for tract in skymap:
        center = tract.getCtrCoord()
        ra = center.getRa().asDegrees()
        dec = center.getDec().asDegrees()

        fields[f"tract_{tract.getId()}"] = {"field_name": f"WFD tract {tract.getId()}", "ra": ra, "dec": dec}

    return fields

### Select the target

In [ ]:
# the_target = lsstcomcam_targets["Seagull"]
# the_target = lsstcomcam_targets["47 Tuc"] # bad
# the_target = lsstcomcam_targets["Fornax dSph"]
# the_target = lsstcomcam_targets["ECDFS"]

# key = "Seagull"
# key = "Fornax dSph"
key = "ECDFS"
# key = "EDFS"
# key = "47 Tuc"
# key = "Rubin SV 38 7"
# key = "Rubin SV 95 -25"

the_target = lsstcam_targets[key]
target_ra = the_target["ra"]
target_dec = the_target["dec"]
target_title = (
    the_target["field_name"] + f" band  {BANDSEL} " + f" (ra,dec) = ({target_ra:.2f},{target_dec:.2f}) "
)
target_point = SpherePoint(target_ra, target_dec, degrees)

In [ ]:
figname_cut = f"MosaicView_{key}"

## Find the list of tract numbers from Object Table

In [ ]:
datasettype = "objectTable_tract"
therefs = butler.registry.queryDatasets(datasettype, collections=collection)
tractsId_list = np.unique([ref.dataId["tract"] for ref in therefs])
tractsId_list = sorted(tractsId_list)
print(tractsId_list)

## Search all deepCoadd

- deepCoadd_calexp comes with WCS

In [ ]:
[d.name for d in butler.registry.queryDatasetTypes() if "coadd" in d.name.lower()]

In [ ]:
coadd_candidates = [
    "deepCoadd",
    "deepCoadd_calexp",
    "deepCoadd.image",
    "deepCoadd_calexp.image",
    "coadd",
    "goodSeeingCoadd",
    "dcrCoadd",
    "deep_coadd",
    "deep_coadd_cell_predetection",
    "template_coadd",
    "pretty_coadd",
]

In [ ]:
for name in coadd_candidates:
    if dataset_type_exists(butler, name):
        print(f"✔ DatasetType exists: {name}")

In [ ]:
# coadd_exp, dataset_used = get_first_existing_dataset(
#    butler,
#    coadd_candidates,
#    dataId,
#    collections="collection"
# )

In [ ]:
# List all  deepCoadd_calexp which are in the butler collection
# Thus all patch and tracts
# refs = butler.registry.queryDatasets("deepCoadd_calexp", collections = collection)
# for ref in refs:
#    print(ref.dataId)

## Find the DataId

In [ ]:
tract_info = skymap.findTract(target_point)
patch_info = tract_info.findPatch(target_point)
bbox = patch_info.getOuterBBox()

print("Patch bounding box:", bbox)

print("Tract ID :", tract_info.getId())
tractNbSel = tract_info.getId()

print("Patch Index :", patch_info.getIndex(), " , ", patch_info.getSequentialIndex())  # (x, y)
print("Bounding Box", bbox)

patchNbSel = patch_info.getSequentialIndex()

In [ ]:
central_patch = patch_info.getIndex()
central_x, central_y = patch_info.getIndex()
neighbor_patches = [
    f"{x},{y}"
    for x in range(central_x - 1, central_x + 2)
    for y in range(central_y - 1, central_y + 2)
    if 0 <= x <= 8 and 0 <= y <= 8
]

In [ ]:
neighbor_patches_indexes = [
    Index2D(x=x, y=y)
    for x in range(central_x - 1, central_x + 2)
    for y in range(central_y - 1, central_y + 2)
    if 0 <= x <= 8 and 0 <= y <= 8
]

In [ ]:
neighbor_patches_indexes

In [ ]:
neighbor_patches_seqindexes = [
    tract_info[patch_index].getSequentialIndex() for patch_index in neighbor_patches_indexes
]

In [ ]:
neighbor_patches_seqindexes

In [ ]:
mapdict_patchesindexes = {}
for patch_index in neighbor_patches_indexes:
    patch_seqindex = tract_info[patch_index].getSequentialIndex()
    mapdict_patchesindexes[patch_seqindex] = f"{patch_index.x},{patch_index.y}"
mapdict_patchesindexes

In [ ]:
# Add the patch and band to the dataId, we didn't need them for the objectTable_tract because it covers all patches and bands
# However the coadds are stored by patch and band dimensions so we have to add them to the dataId

dataId = {"band": BANDSEL, "tract": tractNbSel, "patch": patchNbSel, "skymap": skymapName}

## Fetch the DeepCoadd

In [ ]:
coadd_exp = butler.get("deep_coadd", dataId)

## Plot the (tract,patch) in matplotlib

In [ ]:
image_array = coadd_exp.image.array
image = coadd_exp.image
wcs = coadd_exp.getWcs()
psf = coadd_exp.getPsf()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
im = ax.imshow(image_array, cmap="gray", origin="lower", vmin=0, vmax=2000)
ax.set_title(target_title)
plt.colorbar(im, ax=ax)
plt.show()

In [ ]:
from astropy.wcs import WCS
import matplotlib.pyplot as plt
from astropy.visualization import ZScaleInterval

# Get astropy WCS to plot accordingly
wcs_astropy = WCS(wcs.getFitsMetadata())  # Alternative en extrayant l'entête FITS

# Use zscale to norm
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(image_array)


fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(1, 1, 1, projection=wcs_astropy)
im = ax.imshow(image_array, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)

ax.set_xlabel("RA (deg)")
ax.set_ylabel("Dec (deg)")
ax.coords.grid(True, color="white", ls="dotted")
plt.title("DeepCoadd_calexp for " + target_title)
# plt.colorbar(im, ax=ax)
plt.show()

### Clear memory

In [ ]:
del image_array
del wcs_astropy
del psf
del coadd_exp
gc.collect()

## Check which patches have been generated from Object Table

In [ ]:
# keep a reference toward the objectTable_tract without loading it in memory
refs = butler.registry.queryDatasets("objectTable_tract", collections=collection)
for ref in refs:
    if ref.dataId["tract"] == tractNbSel:
        break

### use butler registry to access to patches

In [ ]:
patches = registry.queryDimensionRecords("patch", dataId={"tract": tractNbSel, "skymap": skymapName})

In [ ]:
listOfProcessedPatches = []
for patch_record in patches:
    # print(patch_record)
    if patch_record.id in neighbor_patches_seqindexes:
        listOfProcessedPatches.append(patch_record.id)
listOfProcessedPatches = sorted(listOfProcessedPatches)
listOfProcessedPatches = np.array(listOfProcessedPatches)

In [ ]:
print(listOfProcessedPatches)

- objectTable_tract is very big. Need to load one by one

In [ ]:
print(
    f"list of geometrically nearby patches from center patch {patchNbSel} in tract {tractNbSel} : ",
    neighbor_patches_seqindexes,
)
print(f"list of processed patches in tract {tractNbSel}", listOfProcessedPatches)

## Plot the Mosaic with Matplotlib per band

In [ ]:
bands = ["u", "g", "r", "i", "z", "y"]

In [ ]:
all_deepCoaddsMosaics = {}
all_titles = {}

# loop on bands
for idx, band in enumerate(bands):
    the_band = band
    the_title = key + f" band {band}"
    print(the_title)
    the_band_titles = []
    try:
        # collection of deepcoadds to build the mosaic
        deepCoaddsMosaicSet = []
        # loop on all patches using the sequential number
        for ipatch in listOfProcessedPatches:
            print(ipatch)
            # build the dataId
            the_dataId = {"band": the_band, "tract": tractNbSel, "patch": ipatch, "skymap": skymapName}

            current_title = the_title + f" tract = {tractNbSel} patch = {ipatch}"

            try:
                # fetch the deepCoadd
                coadd_exp = butler.get("deep_coadd", the_dataId)

                coadd_img = coadd_exp.image  # extrait la MaskedImage
                coadd_binned = binImage(coadd_img, 4)  # applique le binning à l’image
                # add the coadd to the set of coadd
                deepCoaddsMosaicSet.append(coadd_binned.array)
                the_band_titles.append(current_title)
            except Exception as e:
                print(f"Fails with patch {ipatch} : exception = {e}")
        NC = len(deepCoaddsMosaicSet)
        print(f"- got {NC} deepCoadds for band {band}")
        # build the mosaic after gathering all deepCoadds
        # mosaic, mosaic_full = make_mosaic(deepCoaddsMosaicSet, binning=8,camera)
        # print(f"- passed   make_mosaic for band {band}")
        # del mosaic_full
        # gc.collect()
        # print(f"- cleaned mosaic_full for band {band}")
        # save the mosaic for that band
        all_deepCoaddsMosaics[band] = deepCoaddsMosaicSet
        print(f"- added mosaic into all_deepCoaddsMosaics for band {band}")
        # keep the title also
        all_titles[band] = the_band_titles

    except Exception as inst:
        print(f"{key} :: catch Exception for band {band}")
        print(type(inst))  # the exception type
        print(inst.args)  # arguments stored in .args
        print(inst)  # __str_

## Select which band you want to see in firefly

In [ ]:
band_sel = "i"
deepCoaddsMosaics_sel = all_deepCoaddsMosaics[band_sel]
deepCoaddsMosaics_titles_sel = all_titles[band_sel]

figname = figname_cut + f"band_{band_sel}.png"

In [ ]:
len(deepCoaddsMosaics_sel)

In [ ]:
images = deepCoaddsMosaics_sel
titles = deepCoaddsMosaics_titles_sel

In [ ]:
len(images)

In [ ]:
len(titles)

In [ ]:
interval = ZScaleInterval()

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    if i >= len(images):
        ax.axis("off")
        continue
    vmin, vmax = interval.get_limits(images[i])
    ax.imshow(images[i], origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(titles[i], fontsize=10)
    ax.axis("off")

plt.tight_layout()
# plt.suptitle(f"DeepCoadd {band_sel}-band, tract {deepCoaddsMosaics_titles_sel} (mosaïque 3x3)", fontsize=14)
plt.subplots_adjust(top=0.99)
plt.savefig(figname)
plt.show()